#### Simulate poly(A) model

In [3]:
import xarray as xr
import pandas as pd
data = xr.load_dataset("data/white_dataset_mean.nc").sel(time=slice(0, 6))
ds_winata_o = xr.load_dataset("data/ds_winata_polyA_enrich.nc")
ds_winata_r = xr.load_dataset("data/ds_winata_ribo_depletion.nc")
# Gene list
#genes = pd.read_csv("data/increasing_genes_Winata.csv")
genes = pd.read_csv("data/inc_winata_genes.csv")
genes.sort_values("GeneName")

,GeneID,GeneName,Egg,1Cell,16Cells,128c,3.5hpf,5.3hpf,#geneid.1,gene name.1,Egg.1,1Cell.1,16Cells.1,128c.1,3.5hpf.1,5.3hpf.1
21,ENSDARG00000034883,acbd5a,55.267021,38.806279,57.153986,41.082271,40.146125,7.589414,ENSDARG00000034883,acbd5a,12.732921,16.986786,48.117060,47.76430,45.694752,5.114500
0,ENSDARG00000043213,adam17a,37.403300,42.344900,42.341800,34.697500,40.935700,34.597000,ENSDARG00000043213,adam17a,4.281570,5.688180,26.593200,26.86030,35.868700,22.448700
111,ENSDARG00000002912,adipor1a,124.773670,150.338898,114.885748,146.732800,74.127010,16.725187,ENSDARG00000002912,adipor1a,35.568000,40.557500,107.032000,117.10300,111.334000,20.877000
108,ENSDARG00000039429,adka,120.958620,119.465000,170.543348,97.501100,163.452110,117.312010,ENSDARG00000039429,adka,10.622541,10.725000,47.412197,55.61995,68.218740,66.850354
128,ENSDARG00000062272,afg3l2,32.264540,46.499400,36.976370,40.452600,24.018700,2.794105,ENSDARG00000062272,afg3l2,4.896140,3.324560,18.434000,26.33750,26.202434,2.475970
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121,ENSDARG00000003017,zgc:55512,311.168000,264.295000,259.100600,198.545770,264.398400,82.830200,ENSDARG00000003017,zgc:55512,50.043390,52.705000,307.971600,316.80760,310.317200,88.535240
129,ENSDARG00000055162,zhx2,80.579300,72.192600,88.975500,74.390800,70.396000,14.202600,ENSDARG00000055162,zhx2,3.358620,2.410810,32.245700,41.56150,55.826200,8.822630
94,ENSDARG00000020494,znf330,136.112030,126.189770,131.775420,148.266347,68.030740,36.282237,ENSDARG00000020494,znf330,12.760280,26.332270,53.738960,68.06560,58.447800,39.773700
140,ENSDARG00000071868,znf711,78.324300,65.334400,40.795500,48.250300,40.425800,14.969000,ENSDARG00000071868,znf711,3.478240,2.290850,28.744700,37.41600,40.290700,7.730820


In [ ]:
from approx_polyA_model_white_tails import PolyAModel_mean
seed = 1
for gene_id in genes.GeneID.values[0:1]:

    name = genes.loc[genes["GeneID"] == gene_id].GeneName.item()
    model = PolyAModel_mean()
    model.simulate(gene_id, seed, eval=False, plot=True, kernel="nuts", gene_name=name)

0.5111052631578947

# IB model

In [4]:
import ib_polyA_model, ib_capture_bias
from ib_polyA_model import PolyAModel_IB
import numpy as np
import xarray as xr
import arviz as az
import pandas as pd

def IB_simulation(gene_id, gene_name, path="PolyAModel_mean_approx_White_tails"):
        params = pd.read_csv(f"{path}/parameter_fit_summary.csv")
        tails = pd.read_csv("data/Subtelny/tail_lengths.csv")
        transcript = tails.loc[tails["GeneID"] == gene_id, "TranscriptID"].item()

        #fit = xr.open_dataset(f"{path}/{gene_id}/posterior_predictive.nc")
        fit = az.from_netcdf(f"{path}/{gene_id}/numpyro_posterior.nc").posterior_model_fits

        data = xr.load_dataset("data/white_dataset_mean.nc").sel(time=slice(0,6))
        #data_t = xr.load_dataset("data/white_dataset_transcripts.nc").sel(time=slice(0,6))
        obs = data.sel(ensembl_gene_id=gene_id)
        #obs_t = data_t.sel(ensembl_transcript_id=transcript)

        def transcript_length(gene_id):
                """return transcript length and mean length of full dataset [kb]"""
                tpm = pd.read_csv("data/tpms_white_pauli_JN_BK.csv")
                tpm = tpm[tpm["transcript_is_canonical"] == 1.0]
                L_transcript = tpm[tpm["ensembl_gene_id"] == gene_id].transcript_length.item() /1000
                return L_transcript


        model = PolyAModel_IB()

        k_d = params.loc[params["GeneID"] == gene_id, "k_d_mean"].item()
        k_p = params.loc[params["GeneID"] == gene_id, "k_p_mean"].item()

        '''
        inital values for transcript abundance n and polyA tail P0
        '''

        S_depth = 3.8e6
        L_mean_kb = 1.94218
        scaling_factor = 1e6/(S_depth / L_mean_kb)
        L_transcript_kb = transcript_length(gene_id)

        y_max = obs.y.max().item()
        y_0 = obs.y.sel(time=0).item()
        n_transcripts = int(y_max / scaling_factor * L_transcript_kb)

        P0 = round(model.calc_P0(y_0, y_max), 2)
        print("P0: " , P0, "; T0: ", n_transcripts)

        '''
        ---------- Simulation ----------
        '''
        ds_ib = model.simulate(gene_id=gene_id, n_transcripts=n_transcripts, 
                P0_mean=P0, k_p = k_p, k_d = k_d, t_end=6,
                deg_mechanism="PD",)

        '''
        ---------- Plotting ----------
        '''
        #model.plot_functions(t_deg=3)
        ib_capture_bias.plot_capture_results(ds_ib, gene_id)
        ib_capture_bias.plot_tail_distribution(ds_ib, gene_id)
        ib_capture_bias.plot_tail_distribution2(ds_ib, gene_id)

        obs_tails = [
                tails.loc[tails.GeneID == gene_id, "2 hpf"].item(),
                tails.loc[tails.GeneID == gene_id, "4 hpf"].item(),
                tails.loc[tails.GeneID == gene_id, "6 hpf"].item(),
                ]

        ## plot results
        import matplotlib.pyplot as plt
        Phdi = az.hdi(fit, 0.95).P
        TPMhdi = az.hdi(fit, 0.95).TPM_biased

        fit = fit.mean(dim=("chain", "draw"))

        Plower = ds_ib.P_true.min(dim="transcript_id")
        Pupper = ds_ib.P_true.max(dim="transcript_id")

        fig, (ax1, ax2, ax3) = plt.subplots(3,1, figsize=(8,7), height_ratios=[1, 1, 0.5])
        ax1.plot(ds_ib.time, ds_ib.TPM_true, label="debiased (IB)", c="darkgreen", alpha=0.8, ls='--')
        ax1.plot(ds_ib.time, ds_ib.TPM_seq, label="polyA+ biased (IB)", c="darkgreen", alpha=0.8, )

        ax1.plot(fit.time, fit.TPM_true, c="steelblue", label="debiased (ODE)", ls='-.')
        ax1.plot(fit.time, fit.TPM_biased, c="steelblue", label="polyA+ biased (ODE)", )
        ax1.fill_between(fit.time, *TPMhdi.values.T, color="steelblue", alpha=0.1, label="95% hdi")

        ax1.plot(obs.time, obs.y, 'o', label="White et al.", color='k', alpha=0.7)
        #ax1.plot(obs_t.time, obs_t.tpm_values, '^', label="White et al.", color='k', alpha=0.7)

        ax1.legend(loc=(1.02, 0.1), frameon=False)
        ax1.set(title=f"{gene_id} ({gene_name})", xlabel="time (hpf)", ylabel="TPM")

        ax2.plot(ds_ib.time, ds_ib.P_true.mean("transcript_id"),  c="darkgreen", label="polyA-IB-model")
        ax2.fill_between(ds_ib.time, y1=Plower, y2=Pupper,  color="darkgreen", alpha=0.1)
        #ax2.plot(ds_ib.time, Pupper,  c="darkgreen", ls="--", lw=1)

        ax2.plot(fit.time, fit.P,  c="steelblue", label="polyA-ODE-model")
        ax2.fill_between(fit.time, *Phdi.values.T, color="steelblue", alpha=0.2, label="95% hdi")

        ax2.axhline(y=10, xmin=0, xmax=1, c="grey", ls=':', label="$P_{min}$")
        ax2.plot([2, 4, 6], obs_tails,  c="k", marker="x", ms=7, mew=1.5, ls="", label="Subtelny et al.")
        #ax2.axhline(y=15, xmin=0, xmax=1, c="grey", linestyle='dashdot', label="K_b")

        ax2.legend(loc=(1.02, 0.2),  frameon=False)
        ax2.set(title="poly(A) tail length", xlabel="time (hpf)", ylabel="poly(A) tail length (nt)")

        ax3.plot(fit.time, fit.H, c="red")
        ax3.set(title="Degradation hazard H", xlabel="time (hpf)", ylabel="H")

        #ax4.plot(fit.time, fit.miRNA, c="orange")
        #ax4.set(title="Regulator activity", xlabel="time (hpf)", ylabel="")

        plt.tight_layout()
        plt.savefig(f"figures/ib_compare/{gene_id}_model_comparison_TPM.png", dpi=300)
        #plt.show()
        plt.close()
        return ds_ib

In [5]:
genes = ["ENSDARG00000040266", "ENSDARG00000009250",  "ENSDARG00000003250" ]
ds = pd.read_csv("data/inc_winata_genes.csv")
for gene_id in genes:
    name = ds.loc[ds["GeneID"] == gene_id, "GeneName"].item()
    results = IB_simulation(gene_id, name)

P0:  6.36 ; T0:  3063


c:\Users\mgrho\polyA_model\ib_polyA_model.py:51: RuntimeWarning: divide by zero encountered in divide
  return 1 / ( 1 + (t_on/t)**s)


P0:  3.48 ; T0:  75


c:\Users\mgrho\polyA_model\ib_polyA_model.py:51: RuntimeWarning: divide by zero encountered in divide
  return 1 / ( 1 + (t_on/t)**s)


P0:  9.25 ; T0:  328


c:\Users\mgrho\polyA_model\ib_polyA_model.py:51: RuntimeWarning: divide by zero encountered in divide
  return 1 / ( 1 + (t_on/t)**s)


In [6]:
genes = pd.read_csv("data/Subtelny/tail_lengths.csv").GeneID.values
names = pd.read_csv("data/Subtelny/tail_lengths.csv").GeneName.values
na_genes = ["ENSDARG00000070801", "ENSDARG00000100072", "ENSDARG00000053138", "ENSDARG00000038835", "ENSDARG00000071562"]

for i in range(len(genes)):
    
    gene_id = genes[i]
    gene_name = names[i]

    if gene_id in na_genes:
        continue

    print(f"--{i}-- {gene_id}")
    IB_simulation(gene_id, gene_name)

--0-- ENSDARG00000038876
P0:  3.77 ; T0:  86


c:\Users\mgrho\polyA_model\ib_polyA_model.py:51: RuntimeWarning: divide by zero encountered in divide
  return 1 / ( 1 + (t_on/t)**s)


--1-- ENSDARG00000040266
P0:  6.36 ; T0:  3063


c:\Users\mgrho\polyA_model\ib_polyA_model.py:51: RuntimeWarning: divide by zero encountered in divide
  return 1 / ( 1 + (t_on/t)**s)


--2-- ENSDARG00000062272
P0:  4.48 ; T0:  251


...

c:\Users\mgrho\polyA_model\ib_polyA_model.py:51: RuntimeWarning: divide by zero encountered in divide
  return 1 / ( 1 + (t_on/t)**s)


--142-- ENSDARG00000101542
P0:  5.28 ; T0:  31


c:\Users\mgrho\polyA_model\ib_polyA_model.py:51: RuntimeWarning: divide by zero encountered in divide
  return 1 / ( 1 + (t_on/t)**s)


--143-- ENSDARG00000070028
P0:  8.42 ; T0:  276


c:\Users\mgrho\polyA_model\ib_polyA_model.py:51: RuntimeWarning: divide by zero encountered in divide
  return 1 / ( 1 + (t_on/t)**s)
